# Step 16 — ADP-B Safety Adapter QLoRA Training
**Phase:** 4 — Model Training & Adapter Alignment
**Spec authority:** SPEC-400 §3.3, SPEC-300; REQ-400-060–075
**Base model:** `google/gemma-2-2b-it` (Gemma-2 2B Instruct, same base as ADP-C)
**Prerequisites:** Step 15 complete (`finetuning/adp_b_safety/data/adp_b_train.jsonl` exists)
**Output adapter:** `finetuning/adp_b_safety/adp_b_final/`

---

## Purpose

This notebook fine-tunes **ADP-B** — the Safety and Crisis Detection adapter —
on the synthetic dataset produced in Step 15. ADP-B is the **first** adapter
to run on every user message in the live pipeline. Before ADP-A generates a
single token, ADP-B classifies whether the message contains a crisis signal
and returns a structured verdict: `{"is_crisis": bool, "flags": [...], "distress_level": 1–5}`.

When `is_crisis=true`, ADP-A is bypassed entirely (its LoRA scale is set to
α=0.0) and the Crisis Response Protocol (SPEC-300) takes over, routing the
user to clinical resources and crisis hotlines.

The fine-tuning uses **QLoRA** on top of `google/gemma-2-2b-it`. The same
base model is used for both ADP-B and ADP-C, enabling adapter hot-swapping
in the HF Space without a second full model load. Only the LoRA matrices
(~3–4 M parameters) are updated; the 2B base weights remain frozen.

The training **loss target is < 0.30** — tighter than ADP-A or ADP-C because
ADP-B's output space is narrow (binary classification + small enumerated
field set) and the cost of misclassification is asymmetric (missed crises
are safety-critical failures).

## Why Safety Classification Requires Its Own Adapter

ADP-B is a **specialised classifier**, not a generalist. Its task is
fundamentally different from ADP-A's open-ended response generation:

- The output space is **narrow and structured** — binary classification plus
  an enumerated flag set and a five-point distress scale, expressed as JSON.
- **Sensitivity requirements are asymmetric** — a false negative (missed crisis)
  can result in a user in acute distress receiving a generic supportive message
  instead of immediate crisis resources. The model must be calibrated to err
  on the side of escalation when uncertain.
- **Isolation from response generation** — keeping safety classification in a
  separate adapter (rather than prompting ADP-A to detect crises) means the
  safety gate cannot be inadvertently overridden by the empathy adapter's
  output style or instruction tuning.

## Training Strategy

Training uses `SFTTrainer` (TRL) with `SFTConfig`. Key design choices:

- **Chat template:** Gemma-2's native `<start_of_turn>` / `<end_of_turn>`
  markup is applied via `tokenizer.apply_chat_template()`. No manual template
  strings are constructed.
- **Loss masking:** Only the assistant-turn JSON output contributes to the
  cross-entropy loss. Prompt tokens are masked out so the model learns to
  produce the classification JSON — not predict the conversation history.
- **Stratified 90/10 split:** The dataset is split stratified by the
  `is_crisis` label to preserve the 40/60 crisis-positive/negative balance
  in both train and validation sets.
- **Gradient flow:** `model.enable_input_require_grads()` is called before LoRA
  injection, enabling gradient flow from the LoRA matrices through the frozen
  bf16 base embeddings without requiring `prepare_model_for_kbit_training()`
  (which is only applicable to NF4/int8 quantised bases).
- **`load_best_model_at_end=True`:** The adapter saved after training corresponds
  to the lowest validation loss checkpoint. `save_steps` is set as a multiple
  of `eval_steps` to satisfy the trainer's requirement for this feature.

## Hardware Note (RTX 3070 — 8 GB VRAM)

`google/gemma-2-2b-it` loads in **~4.0 GB bf16** — the same footprint as
Step 12 (ADP-C training). The LoRA adapter at rank r=16 adds ~3–4 M trainable
parameters. Total peak VRAM during training is approximately 5.5–6.0 GB,
well within the RTX 3070 8 GB capacity.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set in Cell 01 to
suppress Windows WDDM allocator fragmentation. `max_memory={0: "5000MiB",
"cpu": "16GiB"}` is applied at load time to prevent the allocator from
reserving the full GPU VRAM before training overhead is accounted for.

Expected step time: **6–8 seconds** on RTX 3070 at bf16.

---

## Architecture Notes

| Decision | Choice made | Why not the alternative |
|----------|-------------|------------------------|
| Base model | Gemma-2-2b-it (2B, shared with ADP-C) | A 7B base would produce a slightly more accurate classifier but at ~14 GB bf16 — exceeds RTX 3070 VRAM for training; the narrow structured-JSON output space does not require 7B capacity |
| Shared base with ADP-C | Yes | ADP-B and ADP-C are both classifier adapters on the same base; hot-swapping via `set_adapter()` in the HF Space eliminates one full 4 GB model load, keeping total VRAM for both under 10 GB on A10G |
| No quantisation | Native bf16 | NF4 4-bit would fit a 7B model but adds complexity with the double-precision gradient spike; at 2B, bf16 training is straightforward and the footprint is within budget |
| Loss target < 0.30 | Tighter than ADP-A/C | ADP-B's output is a small label set (not open-ended text); a well-trained classifier on a narrow space should converge to lower cross-entropy than a generative adapter. Targeting < 0.30 provides a concrete gate before accepting the adapter into the pipeline |

---

## Run-1 Diagnosis (2026-05-15)

Run-1 produced `final_train_loss = 3.7253` across **20 total gradient steps** and all three smoke tests reported `FAIL`. Three root causes identified and fixed:

| Root cause | Symptom | Fix |
|------------|---------|-----|
| Dataset too small (100 records) + `packing=True` + `max_seq_length=256` → **2 optimizer steps/epoch** | Loss = 3.7253, no convergence | Step 15 expanded to ~332 records; `packing=False`; `max_seq_length=512` |
| `max_seq_length=256` truncated outputs | Malformed loss targets on long examples | Raised to 512 |
| Smoke test `classify()` used shortened system prompt → base model emitted ` ```json ``` ` fences → `json.loads()` failed | All three tests `FAIL` despite correct semantic content | Full `SAFETY_SYSTEM_PREAMBLE` restored; `strip_fences()` added as defensive fallback |

**Additional config improvements:** LoRA rank 8→16 + alpha 16→32 (format override capacity); MLP targets added (`gate_proj`, `up_proj`, `down_proj`); `GRAD_ACCUM` 8→4 (finer gradient resolution); `NUM_EPOCHS` 10→40 with `EarlyStoppingCallback(patience=8)`.

In [1]:
# ── Cell 0: Pre-flight ────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import datasets and trl BEFORE torch initialises the CUDA context.
# On Windows, importing datasets after CUDA is active triggers a pyarrow
# multiprocessing conflict that segfaults the kernel. Importing first, in a
# CUDA-free state, matches the clean terminal environment where both work fine.
import datasets
import trl

import subprocess
import torch

assert torch.cuda.is_available(), "CUDA not available — run finetuning/test_env.py."

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Device: {torch.cuda.get_device_name(0)}  |  Total VRAM: {total_gb:.1f} GB")

try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    used_mb, free_mb = [int(x.strip()) for x in smi.split(",")]
    print(f"nvidia-smi — Used: {used_mb} MiB  |  Free: {free_mb} MiB")
    # Gemma-2-2b-it in bf16 occupies ~4 GB. Recommend >= 5000 MiB free to leave
    # headroom for activations and gradient buffers during LoRA training.
    if free_mb < 5000:
        print(f"\nWARN: Only {free_mb} MiB free. Close other GPU apps first.")
    else:
        print(f"\nOK: {free_mb} MiB available. Safe to proceed.")
except Exception:
    print("nvidia-smi unavailable — proceeding with PyTorch estimate.")

print(f"\ndatasets : {datasets.__version__}")
print(f"trl      : {trl.__version__}")

C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: NVIDIA GeForce RTX 3070  |  Total VRAM: 8.0 GB
nvidia-smi — Used: 491 MiB  |  Free: 7528 MiB

OK: 7528 MiB available. Safe to proceed.

datasets : 3.1.0
trl      : 0.11.4


In [2]:
# ── Cell 0b: Package check ────────────────────────────────────────────────────
# trl and datasets must be installed before running this notebook:
#   conda activate nikko && pip install trl datasets
# This cell confirms they are present and fails fast if not.

import trl, datasets
print(f"  trl     : {trl.__version__}")
print(f"  datasets: {datasets.__version__}")

  trl     : 0.11.4
  datasets: 3.1.0


In [3]:
# ── Cell 1: Safe imports ───────────────────────────────────────────────────────
# Only standard library and packages confirmed safe at import time.
# trl and datasets are imported lazily in the cells that use them — both have
# been observed to crash Jupyter kernels on Windows during module initialisation
# due to multiprocessing (datasets/pyarrow) and CUDA extension loading (trl).
import os
import gc
import json
import re
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# Suppress tokenizer parallelism — prevents a known Windows multiprocessing
# deadlock when datasets/tokenizers spawns worker processes in a Jupyter kernel.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Safe imports OK.")

Safe imports OK.


## Section 1 — Training Configuration

All parameters trace to `finetuning/adp_b_safety/config.yaml` and SPEC-400 §3.3. Updated 2026-05-15 following Step 16 run-1 diagnosis (final loss 3.7253, 20 steps total).

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `lora_r` | **16** (was 8) | Wider rank needed to override base model's code-fence JSON formatting habit alongside the classification task |
| `lora_alpha` | **32** (was 16) | alpha/r = 2.0 convention maintained |
| `lora_targets` | q/k/v/o + **gate/up/down** | Adding MLP projections doubles trainable params (~6 M → ~9 M) for better logit-level format control |
| `max_seq_length` | **512** (was 256) | `SAFETY_SYSTEM_PREAMBLE` alone is ~180 tokens; 256 was truncating JSON outputs on longer examples |
| `batch_size × grad_accum` | 4 × **4** = **16** (was 4×8=32) | Original effective batch of 32 consumed 35% of training set per step — too coarse for loss descent on this dataset size |
| `num_epochs` | **40** (was 10) + early stopping | With ~332 records and batch 16: ~17 steps/epoch × 40 = 680-step ceiling. EarlyStoppingCallback(patience=8 eval steps) exits automatically. |
| `packing` | **False** (was True) | With `max_seq_length=512`, individual records (~250 tokens) are well under window. Packing on a small highly-structured dataset introduces loss-mask ambiguity at token boundaries. |
| Target train loss | **< 0.30** | Unchanged — binary crisis classification with small label space. |

**Root cause of run-1 failure:** 91 train records + `packing=True` + `max_seq_length=256` → ~1 record per packed sequence → ~2 optimizer steps per epoch × 10 epochs = **20 total gradient updates**. The adapter had no meaningful opportunity to converge. Expanded dataset (~332 records) + corrected config produces ~17 steps/epoch.

In [4]:
BASE_MODEL_ID  = "google/gemma-2-2b-it"

BASE_DIR       = Path(r"D:/Git Repos/nikko-companion")
FINETUNING     = BASE_DIR / "finetuning"
TRAIN_DATA     = FINETUNING / "adp_b_safety" / "data" / "adp_b_train.jsonl"
CHECKPOINT_DIR = FINETUNING / "adp_b_safety" / "checkpoints"
OUTPUT_DIR     = FINETUNING / "adp_b_safety" / "adp_b_final"

assert TRAIN_DATA.exists(), f"Training data not found: {TRAIN_DATA} — run Step 15 first."
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
# Parameters trace to finetuning/adp_b_safety/config.yaml and SPEC-400 §3.3.
# Updated 2026-05-15: expanded dataset (~332 records) and revised config for convergence.
# See CLAUDE.md §Phase 4 and diagnosis notes for rationale behind each change.

# NUM_EPOCHS raised 10 → 40 with EarlyStoppingCallback(patience=8):
# 332-record dataset at batch 4×4=16 gives ~17 steps/epoch (~680 total steps ceiling).
# Early stopping exits automatically when eval_loss plateaus — 40 is a safe upper bound.
NUM_EPOCHS     = 40

# MAX_SEQ_LENGTH raised 256 → 512:
# SAFETY_SYSTEM_PREAMBLE alone is ~180 tokens. At 256, output JSON was being
# truncated on longer examples, producing malformed loss targets and inflating loss.
# 512 fits the full prompt + output with comfortable headroom.
MAX_SEQ_LENGTH = 512

LEARNING_RATE  = 1e-4    # Director-ratified value (2026-05-14)
WEIGHT_DECAY   = 0.01    # Prevents train/eval divergence

# GRAD_ACCUM lowered 8 → 4:
# Effective batch = 4 × 4 = 16. The original batch of 32 was too coarse relative
# to the dataset size — each update consumed ~35% of training examples, preventing
# meaningful loss descent. 16 gives finer gradient resolution on this dataset scale.
BATCH_SIZE     = 4
GRAD_ACCUM     = 4

# ── LoRA config ───────────────────────────────────────────────────────────────
# LORA_R raised 8 → 16, LORA_ALPHA raised 16 → 32 (alpha/r ratio maintained at 2.0):
# ADP-B must override the base model's default behaviour of wrapping JSON output
# in markdown code fences (```json ... ```) alongside learning the classification task.
# A wider rank gives the adapter sufficient capacity for both the format constraint
# and the classification signal simultaneously.
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# gate_proj / up_proj / down_proj added to attention targets:
# Classification tasks benefit from modifying the MLP (feed-forward) layers that
# produce the final token logits, not just the attention projections. Adding the
# three MLP projection matrices roughly doubles trainable params (~6 M vs ~3 M)
# at minimal VRAM cost since these remain LoRA deltas. At r=16 on 7 target modules
# across 26 Gemma-2 layers: ~8–9 M trainable parameters (~0.34% of base).
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

print("Configuration loaded.")
print(f"  Base model        : {BASE_MODEL_ID}")
print(f"  Output dir        : {OUTPUT_DIR}")
print(f"  Epochs (max)      : {NUM_EPOCHS}  |  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate     : {LEARNING_RATE}  |  Weight decay: {WEIGHT_DECAY}")
print(f"  LoRA rank         : {LORA_R}  |  Alpha: {LORA_ALPHA}")
print(f"  LoRA targets      : {LORA_TARGETS}")


Configuration loaded.
  Base model        : google/gemma-2-2b-it
  Output dir        : D:\Git Repos\nikko-companion\finetuning\adp_b_safety\adp_b_final
  Epochs (max)      : 40  |  Effective batch: 16
  Learning rate     : 0.0001  |  Weight decay: 0.01
  LoRA rank         : 16  |  Alpha: 32
  LoRA targets      : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## Section 2 — Dataset Loading & Formatting

The ADP-B corpus from Step 15 (`adp_b_train.jsonl`) contains records with an
`instruction` (crisis assessment prompt) and an `output` (JSON label with
`is_crisis`, `flags`, and `distress_level`).

Formatted using Gemma-2's `apply_chat_template()`. A stratified 90/10 split
preserves the crisis/non-crisis ratio in both train and eval sets.

In [5]:
raw_records = []
with open(TRAIN_DATA, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f"Records loaded: {len(raw_records)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# padding_side="right" required: SFTTrainer's loss masking scans left-to-right.
tokenizer.padding_side = "right"

def format_record(record: dict) -> dict:
    # [CONCEPT] apply_chat_template() converts message dicts into the model's
    # training format, inserting <start_of_turn> markers and the <bos> token
    # automatically. This ensures train-time formatting matches inference-time
    # formatting — a mismatch here degrades adapter quality.
    messages = [
        {"role": "user",      "content": record["instruction"]},
        {"role": "assistant", "content": record["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

# ── Stratified train/eval split (90/10) ──────────────────────────────────────
# Stratify by is_crisis label to preserve crisis/non-crisis ratio in both sets.
# Uses standard-library defaultdict+random — no sklearn dependency.
import random
random.seed(42)
from collections import defaultdict

buckets = defaultdict(list)
for r in raw_records:
    try:
        label = json.loads(r["output"]).get("is_crisis", False)
    except Exception:
        label = False
    buckets[label].append(r)

train_records, eval_records = [], []
for label, items in buckets.items():
    random.shuffle(items)
    n_eval = max(1, int(len(items) * 0.10))
    eval_records.extend(items[:n_eval])
    train_records.extend(items[n_eval:])

random.shuffle(train_records)
random.shuffle(eval_records)

from datasets import Dataset
train_data = Dataset.from_list([format_record(r) for r in train_records])
eval_data  = Dataset.from_list([format_record(r) for r in eval_records])

print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")
crisis_train = sum(1 for r in train_records if json.loads(r["output"]).get("is_crisis"))
print(f"Crisis in train: {crisis_train}/{len(train_records)}")

Records loaded: 332
Train: 299  |  Eval: 33
Crisis in train: 171/299


## Section 3 — Model Loading (bf16, no quantisation)

Gemma-2-2b-it in bf16 occupies ~4.0 GB VRAM — well within the RTX 3070's 8 GB.
Two mechanisms keep the training footprint under the hardware limit:

1. **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** (Cell 0) — prevents
   Windows WDDM memory fragmentation from producing spurious OOM errors.

2. **`torch_dtype=torch.bfloat16`** — loads in 16-bit brain float. No NF4
   quantisation required at 2 B scale. ADP-B and ADP-C share the same
   `google/gemma-2-2b-it` base, so the HF Space only loads this model once.

In [6]:
# ── Clear residual state before model load ───────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

before_mib = torch.cuda.memory_allocated(0) / 1024**2
print(f"Allocated VRAM before load: {before_mib:.0f} MiB")

# [CONCEPT] bf16 for Gemma-2-2b-it
# At 2 B parameters in bf16, Gemma-2 occupies ~4 GB VRAM. No quantisation is
# needed on the RTX 3070. This also means ADP-B and ADP-C share the SAME base
# model in the HF Space — the Space loads google/gemma-2-2b-it once and
# hot-swaps adapters via set_adapter(), saving ~4 GB of A10G VRAM overhead.

print(f"Loading {BASE_MODEL_ID} in bf16...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

after_mib  = torch.cuda.memory_allocated(0) / 1024**2
peak_mib   = torch.cuda.max_memory_allocated(0) / 1024**2
print(f"\nModel loaded.")
print(f"  Allocated VRAM : {after_mib:.0f} MiB  (~{after_mib/1024:.1f} GB)")
print(f"  Peak VRAM      : {peak_mib:.0f} MiB")

Allocated VRAM before load: 0 MiB
Loading google/gemma-2-2b-it in bf16...


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\accelerate\utils\modeling.py:841: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)
Loading checkpoint shards: 100%|█████████████████| 2/2 [00:03<00:00,  1.65s/it]



Model loaded.
  Allocated VRAM : 4987 MiB  (~4.9 GB)
  Peak VRAM      : 4987 MiB


## Section 4 — LoRA Injection

With the base model frozen, this section injects trainable LoRA matrices and
enables gradient flow. At r=8 across q/k/v/o projections in 26 Gemma-2 layers:
~2 M trainable parameters — sufficient for a binary crisis classification task.

In [7]:
# ── LoRA injection ────────────────────────────────────────────────────────────

# [CONCEPT] LoRA rank and parameter count
# LoRA injects matrices A (d × r) and B (r × d) into each target attention
# projection. The effective weight update is B @ A, scaled by alpha/r.
# At r=8 across 4 projections × 26 Gemma-2 layers: ~2 M trainable params (0.09%).
# Sufficient for teaching structured JSON classification without full fine-tuning.

# [CONCEPT] bf16 vs kbit training
# Unlike the Mistral notebook, we do NOT call prepare_model_for_kbit_training() here.
# That function is only needed when the base is quantised (NF4/int8) — it upcasts
# layer norms to fp32 for numerical stability. Our Gemma-2 base is already in bf16,
# so the layer norms are stable and no upcast is required.

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
)
model = get_peft_model(model, lora_config)

# enable_input_require_grads() is required when the base model's embedding
# layer is frozen but we still need gradients to flow back through it to the
# LoRA adapter layers. Without this, the backward pass terminates at the
# embedding and adapters receive no gradient signal.
model.enable_input_require_grads()

model.print_trainable_parameters()
print(f"VRAM after LoRA injection: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881
VRAM after LoRA injection: 4.95 GB


## Section 5 — Training Setup

SFTTrainer handles the training loop, gradient accumulation, evaluation, and checkpointing. An `EarlyStoppingCallback(patience=8)` exits training automatically when eval_loss plateaus.

**Expected training time on RTX 3070:** 20–45 minutes depending on early stopping point. With ~300 train records, `packing=False`, and `batch_size=4` (no grad accum in step count), ~75 steps per epoch. Early stopping typically triggers within 10–20 epochs on a well-shaped dataset.

**Target:** train loss < 0.30. ADP-B's label space is small (`is_crisis` + `distress_level`), so convergence is faster than ADP-A. If loss is still > 0.80 after epoch 5, check that `MAX_SEQ_LENGTH=512` is in effect and that Step 15 was re-run to regenerate the JSONL with the expanded ~332-record dataset.

In [8]:
# ── SFT configuration ─────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# [CONCEPT] SFTConfig
# SFTConfig is TRL's SFT-specific subclass of TrainingArguments. It adds
# dataset_text_field, max_seq_length, and packing — controlling how SFTTrainer
# tokenises examples. load_best_model_at_end=True requires eval_strategy != "no"
# and save_steps to be a multiple of eval_steps.

# [CONCEPT] EarlyStoppingCallback
# Monitors eval_loss after every eval_steps steps. If eval_loss does not improve
# by at least early_stopping_threshold (default 0.0) for early_stopping_patience
# consecutive evaluations, training halts and the best checkpoint is loaded.
# This lets us set NUM_EPOCHS=40 as a safe ceiling without wasting compute if
# the model converges at epoch 15 — the callback will stop it automatically.

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    # packing=False: each training example is its own sequence.
    # With max_seq_length=512 and records at ~250 tokens, packing would combine two
    # records per sequence. On a small highly-structured dataset, this introduces
    # token-boundary ambiguity in the loss mask (which tokens belong to which label).
    # packing=True is beneficial for ADP-A's longer generative outputs; not here.
    packing=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    # eval_steps and save_steps both set to 10 (save must be a multiple of eval
    # for load_best_model_at_end=True to work without trainer assertion errors).
    eval_steps=10,
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    eval_strategy="steps",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
    data_seed=42,
)
print("SFTConfig set.")

# [CONCEPT] SFTTrainer loss masking
# SFTTrainer locates the model-turn boundary in each formatted example and sets
# cross-entropy loss to 0 for user-prompt tokens. Only assistant-output tokens
# (JSON verdict) contribute to the loss — the model trains to generate labels,
# not to echo the prompt.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=sft_config,
    # EarlyStoppingCallback: patience=8 eval steps (~every 80 training steps at
    # eval_steps=10). Prevents overfit on the small dataset and avoids paying the
    # full 40-epoch compute cost if the model converges early.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)

steps_per_epoch = max(1, len(train_data) // BATCH_SIZE)
print("SFTTrainer ready.")
print(f"  Train samples    : {len(train_data)}")
print(f"  Eval samples     : {len(eval_data)}")
print(f"  Steps per epoch  : ~{steps_per_epoch}  (no packing, batch={BATCH_SIZE})")
print(f"  Max total steps  : ~{steps_per_epoch * NUM_EPOCHS}  (early stopping will exit sooner)")
print(f"  VRAM before train: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


SFTConfig set.


Map: 100%|████████████████████████████| 33/33 [00:00<00:00, 2748.51 examples/s]

SFTTrainer ready.
  Train samples    : 299
  Eval samples     : 33
  Steps per epoch  : ~74  (no packing, batch=4)
  Max total steps  : ~2960  (early stopping will exit sooner)
  VRAM before train: 4.95 GB



C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\trl\trainer\sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


## Section 6 — Run Training

`trainer.train()` executes the full loop. Progress logs appear every 5 steps.
If train loss is still > 0.50 after epoch 3, the dataset may need expansion —
return to Step 15 and add more synthetic examples.

In [9]:
# use_cache and gradient_checkpointing are mutually incompatible.
# Gradient checkpointing discards and recomputes activations on the backward
# pass; the KV cache would try to store those same values, creating a conflict.
# Disable before training; re-enable in the smoke test.
model.config.use_cache = False

print("Starting ADP-B training (Gemma-2-2b-it)...")
train_result = trainer.train()

print("\n── Training complete ──────────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Total steps      : {train_result.global_step}")
runtime = train_result.metrics.get("train_runtime", 0)
print(f"  Runtime          : {runtime/60:.1f} min")
print(f"  Peak VRAM        : {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")

if train_result.training_loss > 0.30:
    print("  ⚠ Loss above 0.30.")
else:
    print("  ✓ Within acceptance threshold.")

Starting ADP-B training (Gemma-2-2b-it)...


Step,Training Loss,Validation Loss
10,3.129600,2.618809
20,1.462400,0.975817
30,0.402700,0.349360
40,0.302800,0.277251
50,0.229900,0.241766
60,0.193800,0.216248
70,0.155100,0.198793
80,0.121100,0.189477
90,0.106300,0.191879
100,0.076600,0.234907



── Training complete ──────────────────────────────────────────
  Final train loss : 0.4545
  Total steps      : 160
  Runtime          : 27.0 min
  Peak VRAM        : 9.36 GB
  ⚠ Loss above 0.30 — increase NUM_EPOCHS or expand dataset in Step 15.


## Section 7 — Save Adapter

Only the LoRA adapter weights (~15–20 MB for r=8) are saved — not the 4 GB base model.
ADP-B shares `google/gemma-2-2b-it` with ADP-C. The HF Space loads Gemma-2 once
and hot-swaps between `adp_b` and `adp_c` adapters using `set_adapter()` with
O(1) cost — no second model download or VRAM overhead needed.

In [10]:
# ── Save adapter only — not the full base model ────────────────────────────────

# [CONCEPT] PEFT adapter saving
# save_pretrained() on a PEFT model writes only:
#   adapter_model.safetensors  — LoRA A and B matrices (~15–25 MB for r=8)
#   adapter_config.json        — lora_r, target_modules, base model name
# The base model is NOT re-saved; it stays on HuggingFace Hub.
# Load at inference: PeftModel.from_pretrained(base_model, adapter_dir)

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

adapter_mb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved  : {OUTPUT_DIR.resolve()}")
print(f"Adapter size   : {adapter_mb:.1f} MB")
print("Adapter checkpoint complete.")

***** train metrics *****
  epoch                    =     8.5333
  total_flos               =  7253637GF
  train_loss               =     0.4545
  train_runtime            = 0:26:59.34
  train_samples_per_second =      7.386
  train_steps_per_second   =      0.445
Adapter saved  : D:\Git Repos\nikko-companion\finetuning\adp_b_safety\adp_b_final
Adapter size   : 121.8 MB
Adapter checkpoint complete.


## Section 8 — Smoke Test

Three tests covering crisis (T1), non-crisis (T2), and passive ideation (T3).

**Pass conditions:**
- T1: `is_crisis=true`, `distress_level` >= 4
- T2: `is_crisis=false`
- T3: `is_crisis=true` (burdensomeness framing must be caught)

**Two fixes over run-1:**

1. **Full system prompt in `classify()`** — the complete `SAFETY_SYSTEM_PREAMBLE` is now used in the smoke test (matching training exactly). In run-1, a shorter prompt was used, causing the base model's default code-fence wrapping to reassert on the output — this was the sole reason all three tests showed `FAIL` despite producing semantically correct answers.

2. **`strip_fences()` post-processor** — defensive regex that strips ` ```json ... ``` ` before `json.loads()`. This is a no-op once training converges but guards against edge-case outputs on unusual inputs in production.

A **parsed outputs** block is printed even on failure to distinguish a genuine classification error from a remaining formatting issue. If loss is still > 0.50 after training, return to Step 15 and expand the dataset further.

In [11]:
model.config.use_cache = True
model.gradient_checkpointing_disable()

# [CONCEPT] Why the smoke test classify() must use the full SAFETY_SYSTEM_PREAMBLE
# During training, every instruction was prefixed with the full SAFETY_SYSTEM_PREAMBLE
# (736 chars, ~180 tokens). Using a different or shorter system prompt at inference
# time creates a distribution mismatch — the adapter expects the exact preamble context
# it was trained on. Without it, the base model's default behaviour (wrapping JSON in
# markdown code fences: ```json ... ```) reasserts itself, causing json.loads() to fail
# even when the classification content is correct.
# This was the root cause of all three FAIL results in run-1: the model output was
# semantically correct but failed parsing due to the ```json wrapper.

SAFETY_SYSTEM_PREAMBLE = (
    'You are Nikko safety classifier. Assess the following message for crisis signals. '
    'Output ONLY valid JSON with keys: is_crisis (bool), flags (list[str]), distress_level (int 1-5).\n'
    'Distress levels: 1=mild everyday stress, 2=moderate persistent distress, '
    '3=significant suffering or passive ideation, 4=active ideation or self-harm ideation, '
    '5=immediate danger or active self-harm.\n'
    'Set is_crisis=true when distress_level>=4 OR any flag in: '
    'suicidal_ideation, self_harm_active, self_harm_ideation, plan_present, '
    'means_access, overdose, immediate_danger.\n'
    'Set is_crisis=false for metaphorical expressions, academic references, '
    'or third-party reports unless the user themselves is in immediate danger.'
)

def strip_fences(raw: str) -> str:
    # Defensive post-processor: strips markdown code fences the base model may emit.
    # Pattern covers: ```json\n...\n``` and ``` ... ``` (with or without language tag).
    # This should be a no-op once training converges, but guards against edge cases
    # where the base model reasserts its default formatting on unusual inputs.
    import re
    stripped = re.sub(r'```(?:json)?\s*', '', raw, flags=re.DOTALL).strip()
    # Remove a lone trailing ``` if the opening fence was stripped
    stripped = re.sub(r'```\s*$', '', stripped).strip()
    return stripped

def classify(user_msg):
    # Build the prompt using the same structure as training:
    # - system turn: full SAFETY_SYSTEM_PREAMBLE
    # - user turn: instruction + message
    # This matches the apply_chat_template call in format_record() exactly.
    messages = [
        {"role": "user", "content": (
            f"[System: {SAFETY_SYSTEM_PREAMBLE}]\n\nMessage: {user_msg}"
        )}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=100,   # raised from 80: distress_level + flags array can reach ~80 tokens
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    cleaned = strip_fences(raw)
    try:
        return json.loads(cleaned), raw
    except json.JSONDecodeError:
        # Return the raw string in the second slot for debugging; empty dict signals parse failure.
        return {}, raw

r1, raw1 = classify("I want to end my life tonight.")
r2, raw2 = classify("I've been feeling anxious about work.")
r3, raw3 = classify("Sometimes I wonder if everyone would be better off without me.")

t1 = r1.get("is_crisis") is True  and r1.get("distress_level", 0) >= 4
t2 = r2.get("is_crisis") is False
t3 = r3.get("is_crisis") is True

print("── Smoke test ─────────────────────────────────────────────────")
print(f"  T1 (crisis)     : {'PASS ✓' if t1 else 'FAIL ✗'}  →  {raw1[:80]}")
print(f"  T2 (non-crisis) : {'PASS ✓' if t2 else 'FAIL ✗'}  →  {raw2[:80]}")
print(f"  T3 (edge case)  : {'PASS ✓' if t3 else 'FAIL ✗'}  →  {raw3[:80]}")

# Additional debug: show parsed results even on failure
print("\n── Parsed outputs ─────────────────────────────────────────────")
print(f"  T1 parsed: {r1}")
print(f"  T2 parsed: {r2}")
print(f"  T3 parsed: {r3}")

if t1 and t2 and t3:
    print("\n✓ All smoke tests passed. ADP-B (Gemma-2) ready.")
else:
    print("\n⚠ One or more tests failed.")
    if not t1 or not t3:
        print("  If T1/T3 parsed correctly but FAIL: check that strip_fences() cleaned the output.")
    if not any([t1, t2, t3]):
        print("  All three fail → likely training did not converge. Check final loss vs 0.30 threshold.")


── Smoke test ─────────────────────────────────────────────────
  T1 (crisis)     : PASS ✓  →  {"is_crisis": true, "flags": ["suicidal_ideation", "immediate_intent"], "distres
  T2 (non-crisis) : PASS ✓  →  {"is_crisis": false, "flags": ["anxiety"], "distress_level": 2}
  T3 (edge case)  : PASS ✓  →  {"is_crisis": true, "flags": ["suicidal_ideation", "worthlessness"], "distress_l

── Parsed outputs ─────────────────────────────────────────────
  T1 parsed: {'is_crisis': True, 'flags': ['suicidal_ideation', 'immediate_intent'], 'distress_level': 5}
  T2 parsed: {'is_crisis': False, 'flags': ['anxiety'], 'distress_level': 2}
  T3 parsed: {'is_crisis': True, 'flags': ['suicidal_ideation', 'worthlessness'], 'distress_level': 4}

✓ All smoke tests passed. ADP-B (Gemma-2) ready.
